# NLE-Predictor: Interactive Research Tutorial
### Thermodynamic Intelligence and Kinetic Modeling for Asymmetric Catalysis

**Author:** Asif Raza  
**Repository:** [github.com/asifverse4/Automated-Non-Linear-Effect-NLE-Predictors](https://github.com/asifverse4/Automated-Non-Linear-Effect-NLE-Predictors)

This notebook demonstrates how to use `nle_predictor` to:
1. **Simulate Kagan $\text{ML}_2$ reservoir and active dimer models**.
2. **Analyze catalyst equilibrium speciation** and minor enantiomer trapping.
3. **Fit experimental chiral chromatography data** to extract thermodynamic $\Delta G$ and $K_{\text{ratio}}$.
4. **Model the Noyori DAIB aggregation mechanism** for dialkylzinc additions.
5. **Compute quantitative mathematical diagnostics** (initial slope $S_0$, maximum amplification factor $\alpha_{\text{max}}$, and Area Under Asymmetry).
6. **Apply standard-state concentration corrections** ($1\text{ atm} \to 1\text{ M}$).

## 1. Setup and Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from nle_predictor import (
    NLECalculator,
    NoyoriDAIBModel,
    analyze_nle_diagnostics,
    concentration_correction_1atm_to_1m,
    fit_kagan_reservoir,
    get_solvent_properties,
    load_experimental_csv,
    plot_monomer_speciation,
    plot_nle_with_experimental,
)

print("NLE-Predictor successfully imported!")

## 2. Kagan $\text{ML}_2$ Reservoir Model Simulation
Simulate the classical Kagan reservoir model where dimers are unreactive resting states.

In [ ]:
calc = NLECalculator(temp=298.15)
cat_ee = np.linspace(0.0, 1.0, 101)

# Simulate strong asymmetric amplification ((+)NLE)
res_amp = calc.simulate(cat_ee, delta_g_kcal=-2.0, k_homo=1e4)

# Simulate asymmetric depletion ((-)NLE)
res_dep = calc.simulate(cat_ee, delta_g_kcal=+1.5, k_homo=1e4)

print(f"Amplification classification: {res_amp.classification}")
print(f"Product ee at 50% catalyst ee: {res_amp.product_at_50 * 100:.2f}%")
print(f"Depletion classification: {res_dep.classification}")

In [ ]:
plt.figure(figsize=(6.5, 5))
plt.plot(cat_ee * 100, cat_ee * 100, 'k--', label='Ideal Linear (No Aggregation)')
plt.plot(cat_ee * 100, res_amp.ee_product * 100, color='#08519c', lw=2.5, label=r'\Delta G = -2.0 kcal/mol ((+)NLE)')
plt.plot(cat_ee * 100, res_dep.ee_product * 100, color='#cb181d', lw=2.5, label=r'\Delta G = +1.5 kcal/mol ((-)NLE)')
plt.title('Kagan ML$_2$ Reservoir Model', fontweight='bold')
plt.xlabel('Catalyst $ee$ (%)', fontweight='bold')
plt.ylabel('Product $ee$ (%)', fontweight='bold')
plt.legend(framealpha=0.9)
plt.grid(True, linestyle=':', alpha=0.6)
plt.show()

## 3. Catalyst Equilibrium Speciation
Analyze how the heterochiral dimer $[(R,S)]$ sequesters the minor enantiomer to purify the active monomer pool.

In [ ]:
plot_monomer_speciation(res_amp)
plt.show()

## 4. Fitting Experimental Data Pairs
Load historical benchmark enantiomeric excess data (Kagan, 1986) and extract experimental $\Delta G$.

In [ ]:
exp_cat, exp_prod = load_experimental_csv('examples/benchmarks/datasets/kagan_amplification.csv')

fit_res = fit_kagan_reservoir(exp_cat, exp_prod, temp=298.15)

print("=" * 50)
print(f"Fitted Delta G (RS - RR): {fit_res.delta_g_kcal:.3f} ± {fit_res.delta_g_stderr:.3f} kcal/mol")
print(f"Fitted K_hetero / K_homo: {fit_res.k_ratio:.2f} ± {fit_res.k_ratio_stderr:.2f}")
print(f"Goodness of Fit (R^2):    {fit_res.r_squared:.4f}")
print(f"RMSE:                    {fit_res.rmse * 100:.2f}%")
print("=" * 50)

In [ ]:
# Simulate dense fitted curve
fitted_sim = calc.simulate(np.linspace(0, 1, 101), delta_g_kcal=fit_res.delta_g_kcal)
plot_nle_with_experimental(fitted_sim, exp_cat=exp_cat, exp_prod=exp_prod, title=f'Kagan Diethylzinc Fit (R^2 = {fit_res.r_squared:.3f})')
plt.show()

## 5. Noyori DAIB Aggregation Model
Simulate dialkylzinc addition catalyzed by DAIB where $K_d(\text{hetero}) \ll K_d(\text{homo})$.

In [ ]:
noyori = NoyoriDAIBModel(ee_0=0.98, kd_homo=1e-3, kd_hetero=1e-6)
res_noyori = noyori.simulate(cat_ee)

print(f"Noyori Model Classification: {res_noyori.classification}")
print(f"Product ee at 10% catalyst ee: {np.interp(0.10, cat_ee, res_noyori.ee_product) * 100:.2f}%")

## 6. Mathematical Diagnostics Profiling

In [ ]:
diag = analyze_nle_diagnostics(res_amp.ee_catalyst, res_amp.ee_product)

print(f"Initial Slope at Racemic S_0: {diag.initial_slope:.3f}")
print(f"Maximum Amplification Factor:  {diag.max_amplification:.3f}x")
print(f"Catalyst ee at Max Amp:       {diag.catalyst_ee_at_max_amplification * 100:.1f}%")
print(f"Area Under Asymmetry (AUA):   {diag.area_under_asymmetry:.4f}")

## 7. Solution Standard-State Concentration Shift
Compute standard-state concentration shifts from gas-phase ($1\text{ atm}$) to solution-phase ($1\text{ M}$).

In [ ]:
corr = concentration_correction_1atm_to_1m(temp_k=298.15, delta_n=-1)
print(f"Dimerization standard-state shift (298.15 K): +{corr:.3f} kcal/mol")

tol = get_solvent_properties('toluene')
print(f"Toluene dielectric constant: {tol.dielectric_constant}, CPCM alias: {tol.orca_cpcm_name}")